# 🔬 Post-Training: KTO
Aligns the SFT model using **KTO**.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../01_sft/qwen_medical_arabic_lora",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)
print("Model loaded for KTO. VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")


In [ ]:
from datasets import load_dataset
import os

DATA_FILE = "../../data/alignment/05_kto_dataset.json"
assert os.path.exists(DATA_FILE), f"Data file not found: {DATA_FILE}"

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Dataset loaded: {len(dataset)} examples")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_kto(example):
    prompt     = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    completion = f"{example['completion']}<|im_end|>\n"
    return {"prompt": prompt, "completion": completion, "label": example["label"]}

dataset = dataset.map(format_kto, remove_columns=dataset.column_names)
pos = sum(1 for x in dataset if x["label"])
print(f"Formatted. Positive: {pos} / Negative: {len(dataset)-pos}")


In [ ]:
from trl import KTOTrainer, KTOConfig

training_args = KTOConfig(
    output_dir          = "outputs_kto",
    beta                = 0.1,
    desirable_weight    = 1.0,
    per_device_train_batch_size = 2,  # KTO requires batch_size > 1
    gradient_accumulation_steps = 4,  # Total batch size remains 8
    learning_rate       = 5e-5,
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.1,
    max_length          = 1024,
    max_prompt_length   = 512,
    optim               = "adamw_8bit",
    fp16                = not is_bfloat16_supported(),
    bf16                = is_bfloat16_supported(),
    num_train_epochs    = 2,
    logging_steps       = 10,
    save_steps          = 50,
    save_total_limit    = 2,
    report_to           = "none",
)

trainer = KTOTrainer(
    model        = model,
    ref_model    = None,
    args         = training_args,
    train_dataset= dataset,
    tokenizer    = tokenizer,
)
print("KTO Trainer ready!")


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("qwen_medical_arabic_kto")
tokenizer.save_pretrained("qwen_medical_arabic_kto")
print("Saved KTO model -> qwen_medical_arabic_kto/")
